### Tools

In [23]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model= init_chat_model("groq:openai/gpt-oss-120b")
response= model.invoke("why do we have seasons on earth?")
response

AIMessage(content='**Short answer:**  \nWe have seasons because the Earth’s axis is tilted relative to its orbit around the Sun. That tilt makes different parts of the planet receive more direct sunlight at different times of the year.\n\n---\n\n## The key ingredients\n\n| Factor | What it is | How it creates seasons |\n|--------|------------|------------------------|\n| **Axial tilt (obliquity)** | The Earth’s spin axis is tilted about **23.5°** away from a line perpendicular to its orbital plane. | When a hemisphere is tilted toward the Sun, sunlight hits it more directly and the days are longer → warmer (summer). When it’s tilted away, sunlight is slanted and days are shorter → cooler (winter). |\n| **Orbit around the Sun** | Earth travels in an almost circular path that takes one year (≈365.24\u202fdays). | As Earth moves, the orientation of the tilt relative to the Sun changes, so each hemisphere alternately points toward and away from the Sun over the year. |\n| **Sun‑angle & day

In [24]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."
@tool
def get_balance():
    """give the balance"""
    return "The balance of the user is $1000."
tools = {
    "get_weather": get_weather,
    "get_balance": get_balance
}

model_with_tools = model.bind_tools([get_weather, get_balance])

In [25]:
response = model_with_tools.invoke("what is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"\nTool called: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks for weather in New York. Use function get_weather.', 'tool_calls': [{'id': 'fc_437deec4-e002-45d8-b8d6-fa0847b9d581', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 145, 'total_tokens': 187, 'completion_time': 0.086689957, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007184264, 'prompt_tokens_details': None, 'queue_time': 0.375323213, 'total_time': 0.093874221}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a010c1-d2ab-7d43-b4fc-fa482f50e0ec-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_437deec4-e002-45d8-b8d6-fa0847b9d581', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input

### Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
  # Execute the tool with the generated arguments
  tool_name = tool_call["name"]
  tool = tools[tool_name]
  tool_result = get_weather.invoke(tool_call)
  messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


content='' additional_kwargs={'reasoning_content': 'The user asks for weather in Boston. We have a function get_weather. Use it.', 'tool_calls': [{'id': 'fc_264b3c2f-f026-4fe3-b97a-a901b6e87b88', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 143, 'total_tokens': 189, 'completion_time': 0.099079096, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.005485622, 'prompt_tokens_details': None, 'queue_time': 0.287289187, 'total_time': 0.104564718}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a010c3-ba09-7053-b4f8-9f178af32487-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_264b3c2f-f026-4fe3-b97a-a901b6e87b88', 'type': 'tool_call'}] invalid_tool_calls=[] usage_meta